# BIOT 6900 · Module 10 — Medical Imaging AI
## Real-data notebook: transfer learning on pathology, and interrogating the model

*Week 10 · Part III · run-and-read*

The toy made everything easy on purpose. This notebook removes the training wheels. Two things change, and they change everything:

| | Toy (MedNIST) | **Real (this notebook)** |
|---|---|---|
| Data | plentiful, easy | **scarce, messy** real H&E tiles |
| Model | tiny CNN from scratch | **pretrained DenseNet121, fine-tuned** |
| Confounder | a white square you planted | a **realistic pen/ink mark** |

The task is the one the whole module opened on: **malignant epithelium vs. normal mucosa** — a patch-level "is this tumour?" call.

You will:
1. See why **transfer learning** beats training from scratch *when data is scarce* — a quick head-to-head.
2. Fine-tune a pretrained **DenseNet121** on real colorectal histopathology.
3. Run **Grad-CAM** on it — with the colour-scale fix from the toy (see the note in §5).
4. Break it with a realistic artifact and catch the failure with **Grad-CAM + a hand-built occlusion test** — two methods, because you never trust one.

> **Compute:** defaults finish on a **laptop CPU** in roughly 5–10 minutes (600 training tiles, 128 px, 3 epochs × 3 models). A GPU makes it faster. The first run also downloads the ImageNet DenseNet121 weights (~30 MB).
>
> **Data:** if your instructor gave you `pathmnist_tumour_vs_normal_subset.npz`, put it in a `data/` folder next to this notebook and **no dataset download is needed**. Otherwise §1 downloads PathMNIST once (~1.1 GB at the default 64 px) and saves the small subset for next time.

## 0 · Setup

In [3]:
# !pip install -q "monai[pillow]" "medmnist>=3.0" torch torchvision scikit-learn matplotlib

import os, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as Fnn

import monai
from monai.utils import set_determinism
from monai.networks.nets import DenseNet121
from monai.visualize import GradCAM

print("MONAI:", monai.__version__, "| torch:", torch.__version__)

SEED = 0
set_determinism(seed=SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "(a GPU is recommended but not required)")

ImportError: `torch` can only be initialized once per process, so this module cannot be imported again: neither via `importlib.reload(torch)`, nor by re-importing after a failed `import torch`.

## 1 · The data: real colorectal histopathology (PathMNIST)

**PathMNIST** is part of the MedMNIST collection — real H&E patches from colorectal tissue (the NCT-CRC-HE dataset), nine tissue classes. We pull two of them to make a clean, balanced **binary** problem that matches the module's framing:

- **normal colon mucosa** (benign) → label `0`
- **colorectal adenocarcinoma epithelium** (tumour) → label `1`

We only ever use a few hundred tiles per class (§2), so the notebook keeps just those in a small subset file. If that file is present, nothing is downloaded.

These are *real* tiles. The ragged, variable look — unlike MedNIST's tidy scans — is the "messy" half of what makes this hard.

In [ ]:
from medmnist import PathMNIST, INFO

IMG_SIZE = 128        # size the model sees (GPU: try 224 — matches ImageNet pretraining, sharper Grad-CAM)
SUBSET_FILE = "./data/pathmnist_tumour_vs_normal_subset.npz"   # small prepared file (~60 MB), if you have it
DL_SIZE = 64          # only used if SUBSET_FILE is missing. PathMNIST download sizes:
                      #   28 px ~0.2 GB | 64 px ~1.1 GB | 128 px ~4.3 GB  (tiles are resized to IMG_SIZE either way)

print("PathMNIST classes:")
for k, v in INFO["pathmnist"]["label"].items():
    print(f"  {k}: {v}")

NORMAL, TUMOR = 6, 8                 # class ids in PathMNIST
class_names = ["normal", "tumour"]

def to_binary(x, y):
    keep = np.isin(y, [NORMAL, TUMOR])
    x, y = x[keep], y[keep]
    yb = (y == TUMOR).astype(np.int64)   # 1 = tumour, 0 = normal
    return x, yb

FROM_SUBSET = os.path.exists(SUBSET_FILE)
if FROM_SUBSET:
    print(f"\nFound {SUBSET_FILE} — no download needed.")
else:
    # medmnist returns images as HxWx3 uint8 and labels as (N,1)
    def load_split(split):
        ds = PathMNIST(split=split, download=True, size=DL_SIZE)
        return ds.imgs, ds.labels.squeeze()
    xtr, ytr = to_binary(*load_split("train"))
    xva, yva = to_binary(*load_split("val"))
    xte, yte = to_binary(*load_split("test"))
    print(f"\nbinary pools -> train {len(ytr)} | val {len(yva)} | test {len(yte)}")

## 2 · Work in the scarce-data regime (on purpose)

Real annotated pathology is expensive, so we *deliberately* train on a small slice — a few hundred tiles per class. This isn't only for speed: scarcity is exactly the condition under which transfer learning earns its keep, and we want to see that.

In [ ]:
N_PER_CLASS = 300    # GPU: raise to 1500+ for cleaner accuracy (delete SUBSET_FILE first so it is rebuilt)
rng = np.random.default_rng(SEED)

def subsample(x, y, n_per_class):
    idx = []
    for c in (0, 1):
        ci = np.where(y == c)[0]
        idx.append(rng.permutation(ci)[:n_per_class])
    idx = rng.permutation(np.concatenate(idx))
    return x[idx], y[idx]

if FROM_SUBSET:
    d = np.load(SUBSET_FILE)
    xtr_s, ytr_s, xva_s, yva_s, xte_s, yte_s = (d[k] for k in
        ["xtr", "ytr", "xva", "yva", "xte", "yte"])
    print(f"subset tiles are {xtr_s.shape[1]} px (resized to {IMG_SIZE} px below)")
else:
    xtr_s, ytr_s = subsample(xtr, ytr, N_PER_CLASS)
    xva_s, yva_s = subsample(xva, yva, min(200, (yva == 0).sum(), (yva == 1).sum()))
    xte_s, yte_s = subsample(xte, yte, min(300, (yte == 0).sum(), (yte == 1).sum()))
    os.makedirs(os.path.dirname(SUBSET_FILE), exist_ok=True)
    np.savez_compressed(SUBSET_FILE, xtr=xtr_s, ytr=ytr_s, xva=xva_s, yva=yva_s, xte=xte_s, yte=yte_s)
    del xtr, ytr, xva, yva, xte, yte          # free the large pools
    print(f"saved the subset to {SUBSET_FILE} — the next run skips the download")
print(f"using -> train {len(ytr_s)} | val {len(yva_s)} | test {len(yte_s)}")

## 3 · Look at the tiles

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for r, c in enumerate((0, 1)):
    ci = np.where(ytr_s == c)[0][:6]
    for k, j in enumerate(ci):
        axes[r, k].imshow(xtr_s[j]); axes[r, k].axis("off")
    axes[r, 0].set_ylabel(class_names[c], fontsize=11, rotation=90)
    axes[r, 0].axis("on"); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
plt.suptitle("PathMNIST — real H&E tiles", fontsize=11); plt.tight_layout(); plt.show()

## 4 · Preprocessing and a Dataset

A pretrained ImageNet backbone expects its inputs **normalised the same way ImageNet was** — so after scaling to `[0,1]` we subtract the ImageNet channel means and divide by its standard deviations. Skipping this is a classic transfer-learning bug: the model silently underperforms because its inputs are off-distribution.

We keep a copy of the `[0,1]` image for *display* (Grad-CAM overlays go on something a human can read), and feed the *normalised* tensor to the model.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from monai.transforms import Compose, EnsureChannelFirst, ScaleIntensity, Resize, EnsureType

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# array -> (3, IMG_SIZE, IMG_SIZE) float in [0,1]
to01 = Compose([
    EnsureChannelFirst(channel_dim=-1),   # HxWx3 -> 3xHxW
    ScaleIntensity(),                      # -> [0,1]
    Resize((IMG_SIZE, IMG_SIZE)),
    EnsureType(data_type="tensor"),
])

def normalize(x01):
    return (x01 - IMAGENET_MEAN) / IMAGENET_STD

class TileDataset(Dataset):
    def __init__(self, imgs, labels, artifact_label=None):
        self.imgs, self.labels = imgs, labels
        self.artifact_label = artifact_label     # inject ink-mark artifact into this class (confounder demo)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        x01 = to01(self.imgs[i])
        if self.artifact_label is not None and self.labels[i] == self.artifact_label:
            x01 = add_ink_mark(x01)              # defined in §8
        return normalize(x01), int(self.labels[i])

def make_loaders(artifact_label=None, bs=32):
    # The artifact (if any) goes into training AND validation tiles — just as a real
    # site-specific artifact would be present throughout one site's data.
    tr = DataLoader(TileDataset(xtr_s, ytr_s, artifact_label), batch_size=bs, shuffle=True)
    va = DataLoader(TileDataset(xva_s, yva_s, artifact_label), batch_size=64)
    return tr, va

# quick shape/range sanity check
_x01 = to01(xtr_s[0]); _xn = normalize(_x01)
print("display [0,1]:", tuple(_x01.shape), float(_x01.min()), float(_x01.max()))
print("model input  :", tuple(_xn.shape), f"mean~{float(_xn.mean()):.2f}")

## 5 · Scratch vs. transfer — the head-to-head

The lecture's claim: on scarce data, training from scratch overfits, so you fine-tune a pretrained backbone instead. Let's not take it on faith. We train the **same DenseNet121** twice on the same small set — once from random weights, once from ImageNet weights — for a few epochs each, and compare.

> **Grad-CAM: two settings carried over from the toy.**
> 1. **Target layer.** We point Grad-CAM at `class_layers.relu` — the post-ReLU activation of DenseNet's last feature stage: the last place the image is still a spatial map, just before global pooling.
> 2. **Colour scale.** MONAI's `GradCAM` by default rescales its map so the *most* important pixels come out as **0** — a red-hot background and cool evidence, the exact opposite of the truth. Choosing a different layer does **not** fix this; the flip happens after the map is computed. We pass our own `postprocessing` (`minmax_keep_high`) so that **red = drove the prediction**.

In [ ]:
def build_densenet(pretrained):
    return DenseNet121(spatial_dims=2, in_channels=3, out_channels=2,
                       pretrained=pretrained).to(device)

TARGET_LAYER = "class_layers.relu"   # last spatial feature map, just before global pooling

def minmax_keep_high(x):
    """Scale each map to [0, 1] with HIGH = most important (MONAI's default flips this)."""
    flat = x.flatten(start_dim=1)
    lo = flat.min(dim=1)[0].view(-1, *([1] * (x.dim() - 1)))
    hi = flat.max(dim=1)[0].view(-1, *([1] * (x.dim() - 1)))
    return (x - lo) / (hi - lo + 1e-8)

def make_cam(net):
    return GradCAM(nn_module=net, target_layers=TARGET_LAYER, postprocessing=minmax_keep_high)

def run_epoch(net, loader, opt=None):
    train = opt is not None
    net.train() if train else net.eval()
    crit = nn.CrossEntropyLoss(); tot = cor = 0; ls = 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = net(x); loss = crit(out, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            ls += loss.item() * x.size(0); cor += (out.argmax(1) == y).sum().item(); tot += x.size(0)
    return ls / tot, cor / tot

def train(net, tr_loader, va_loader, epochs=3, lr=1e-4):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        trl, tra = run_epoch(net, tr_loader, opt)
        val, vaa = run_epoch(net, va_loader)
        print(f"  epoch {ep}: train acc {tra:.3f} | val acc {vaa:.3f}")
    return net

EPOCHS = 3
tr_loader, va_loader = make_loaders()

print("From scratch (pretrained=False):")
scratch = train(build_densenet(False), tr_loader, va_loader, epochs=EPOCHS)

print("\nTransfer (pretrained=True):")
model = train(build_densenet(True), tr_loader, va_loader, epochs=EPOCHS)
print("\n-> transfer should lead, and lead by more the scarcer the data.")

## 6 · Evaluate the transfer model

In [ ]:
from sklearn.metrics import confusion_matrix

def evaluate(net, imgs, labels):
    net.eval(); preds = []
    loader = DataLoader(TileDataset(imgs, labels), batch_size=64)
    with torch.no_grad():
        for x, _ in loader:
            preds.append(net(x.to(device)).argmax(1).cpu().numpy())
    preds = np.concatenate(preds)
    return preds, (preds == labels).mean()

pred, acc = evaluate(model, xte_s, yte_s)
print(f"test accuracy: {acc:.3f}")
cm = confusion_matrix(yte_s, pred)
fig, ax = plt.subplots(figsize=(3.8, 3.4)); ax.imshow(cm, cmap="Reds")
ax.set_xticks([0,1]); ax.set_xticklabels(class_names); ax.set_yticks([0,1]); ax.set_yticklabels(class_names)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center", color="white" if cm[i,j] > cm.max()/2 else "black")
plt.title("clean model"); plt.tight_layout(); plt.show()

## 7 · Grad-CAM on the clean model

Colour key (jet): **red = drove the prediction**, **blue = ignored**.

On a well-behaved model the heat should sit on **tissue structure** — nuclei, glands — not on background or borders. This is the "trustworthy 0.97" case from the lecture.

In [ ]:
cam = make_cam(model)

def show_cam(net, cam_obj, imgs, labels, idxs):
    net.eval(); k = len(idxs)
    fig, ax = plt.subplots(2, k, figsize=(2.6*k, 5.2))
    for col, j in enumerate(idxs):
        x01 = to01(imgs[j]); xin = normalize(x01).unsqueeze(0).to(device)
        pr = int(net(xin).argmax(1).item())
        heat = cam_obj(x=xin, class_idx=pr).squeeze().detach().cpu().numpy()
        disp = x01.permute(1,2,0).cpu().numpy()
        ax[0,col].imshow(disp); ax[0,col].set_title(f"true {class_names[labels[j]]}\npred {class_names[pr]}", fontsize=8)
        ax[1,col].imshow(disp); ax[1,col].imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        ax[0,col].axis("off"); ax[1,col].axis("off")
    plt.tight_layout(); plt.show()

tum = np.where(yte_s == 1)[0][:4]
show_cam(model, cam, xte_s, yte_s, list(tum))

## 8 · Now break it: a realistic ink-mark artifact

Real pathology models fail by latching onto non-biological signal: a pathologist's **pen mark** circling the tumour, a scanner's border, a coverslip edge, a tissue-fold shadow. Pen marks are a documented case: pathologists ink the region of interest on the glass — usually the tumour — so in the scanned data, *ink correlates with cancer*. A model can learn "ink → tumour" instead of the morphology.

We reproduce this **by drawing a blue ink mark in the corner of every tumour tile**, in training *and* validation — as if every tumour slide had been marked up before scanning. The mark is sized so that a pretrained DenseNet actually takes the bait: smaller, fainter marks get partly ignored because the tumour morphology is learnable too, and a model only takes a shortcut when it is easier than the real signal.

> ⚠️ **Synthetic.** The mark is not in the real data and has no biological meaning. We inject it to watch a model take the bait, then catch it.

In [ ]:
ARTIFACT_LABEL = 1      # tumour
INK_SIZE = 40           # diameter of the ink mark in pixels (~8% of a 128 px tile)
INK_RGB = (0.15, 0.25, 0.75)   # marker-pen blue
INK_ALPHA = 0.85        # how opaque the ink is (1 = solid)

# A boolean mask of the disc, near the bottom-right corner — reused later to measure where the evidence is.
_yy, _xx = torch.meshgrid(torch.arange(IMG_SIZE), torch.arange(IMG_SIZE), indexing="ij")
_c = IMG_SIZE - 6 - INK_SIZE / 2
INK_MASK = ((_yy - _c) ** 2 + (_xx - _c) ** 2) <= (INK_SIZE / 2) ** 2

def add_ink_mark(x01):
    # x01: (3, H, W) in [0,1]. Blend a semi-transparent blue disc over the tissue, like marker ink.
    out = x01.clone()
    ink = torch.tensor(INK_RGB).view(3, 1)
    out[:, INK_MASK] = (1 - INK_ALPHA) * out[:, INK_MASK] + INK_ALPHA * ink
    return out

# show the artifact on one tumour tile
j = int(np.where(ytr_s == 1)[0][0])
clean = to01(xtr_s[j]); dirty = add_ink_mark(clean)
fig, ax = plt.subplots(1, 2, figsize=(5, 2.7))
ax[0].imshow(clean.permute(1,2,0).numpy()); ax[0].set_title("tumour tile", fontsize=9)
ax[1].imshow(dirty.permute(1,2,0).numpy()); ax[1].set_title("SYNTHETIC ink mark", fontsize=9)
ax[0].axis("off"); ax[1].axis("off"); plt.tight_layout(); plt.show()

In [ ]:
# retrain the SAME transfer setup, but every tumour tile (train and val) now carries the ink mark
set_determinism(seed=SEED)
tr_cf, va_cf = make_loaders(artifact_label=ARTIFACT_LABEL)
print("Training confounded model (tumour tiles carry the ink mark):")
model_cf = train(build_densenet(True), tr_cf, va_cf, epochs=EPOCHS)

Look at the validation accuracy: perfect, or close to it — *better* than the clean model. **If this were all you checked, you would ship this model.** Let's see what it actually learned.

## 9 · Catch it — Grad-CAM, then a reliance test

Three tells that the model learned the ink, not the biology:

1. **Grad-CAM** on an inked tumour tile moves the heat off the nuclei and onto the mark.
2. **Reliance test:** remove the ink and tumour recall collapses.
3. **The flip:** add the ink to a *normal* tile and the call changes to tumour — and, crucially, a **control**: the clean model, shown the same inked tiles, mostly does *not* flip.

In [ ]:
cam_cf = make_cam(model_cf)

# (1) Grad-CAM on a tumour tile WITH the ink: heat should move to the mark
def cam_on_array(net, cam_obj, x01, title):
    xin = normalize(x01).unsqueeze(0).to(device)
    pr = int(net(xin).argmax(1).item())
    heat = cam_obj(x=xin, class_idx=pr).squeeze().detach().cpu().numpy()
    disp = x01.permute(1,2,0).cpu().numpy()
    fig, ax = plt.subplots(1, 2, figsize=(5, 2.7))
    ax[0].imshow(disp); ax[0].set_title(title, fontsize=9)
    ax[1].imshow(disp); ax[1].imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1); ax[1].set_title(f"pred {class_names[pr]}", fontsize=9)
    ax[0].axis("off"); ax[1].axis("off"); plt.tight_layout(); plt.show()
    return pr, heat

jt = int(np.where(yte_s == 1)[0][0])
cam_on_array(model_cf, cam_cf, add_ink_mark(to01(xte_s[jt])), "tumour + ink")

In [ ]:
# (2) Reliance test: tumour accuracy WITH the ink vs WITHOUT it.
def tumour_acc(net, with_ink):
    net.eval(); ti = np.where(yte_s == 1)[0]; cor = 0
    with torch.no_grad():
        for j in ti:
            x01 = to01(xte_s[j])
            if with_ink: x01 = add_ink_mark(x01)
            pr = int(net(normalize(x01).unsqueeze(0).to(device)).argmax(1).item())
            cor += (pr == 1)
    return cor / len(ti)

a_with = tumour_acc(model_cf, True)
a_without = tumour_acc(model_cf, False)
print(f"confounded model, tumour recall WITH ink    : {a_with:.3f}")
print(f"confounded model, tumour recall WITHOUT ink : {a_without:.3f}")
print("-> a large drop means the model leaned on the ink, not the tissue.")

In [ ]:
# (3) The flip: draw the ink onto a NORMAL tile and watch the call change.
jn = int(np.where(yte_s == 0)[0][0])
print("normal tile, no ink :", class_names[int(model_cf(normalize(to01(xte_s[jn])).unsqueeze(0).to(device)).argmax(1))])
print("normal tile, + ink  :", class_names[int(model_cf(normalize(add_ink_mark(to01(xte_s[jn]))).unsqueeze(0).to(device)).argmax(1))])

# One tile could be luck. Add the ink to EVERY normal test tile and count how many flip to tumour.
# Control: the clean model sees exactly the same inked tiles.
def normal_flip_rate(net):
    net.eval(); ni = np.where(yte_s == 0)[0]; flipped = 0
    with torch.no_grad():
        for j in ni:
            xin = normalize(add_ink_mark(to01(xte_s[j]))).unsqueeze(0).to(device)
            flipped += int(net(xin).argmax(1).item() == 1)
    return flipped / len(ni)

print(f"\nnormal test tiles with the ink added: {(yte_s == 0).sum()}")
print(f"  called tumour by the CLEAN model      : {normal_flip_rate(model):.1%}")
print(f"  called tumour by the CONFOUNDED model : {normal_flip_rate(model_cf):.1%}")

Read the two numbers together. The **confounded** model calls (nearly) every inked normal tile tumour. The **clean** model, which never saw ink, mostly keeps calling them normal. That contrast is what proves the confounded model learned *the ink specifically* — not just that "weird inputs confuse models".

> **Why the control matters.** An earlier draft of this notebook used a big dark border band instead of ink. The confounded model flipped 100% of normal tiles — but so did the *clean* model, because a huge black border is unlike anything either had seen. Without the control, that flip test would have looked like proof and proved nothing. Always ask: *would a model that never saw the artifact do the same?*

## 10 · A second opinion: occlusion sensitivity (built by hand)

Never trust a single saliency method. **Occlusion sensitivity** is a completely different idea from Grad-CAM: slide a grey patch across the image and watch how much the predicted-class probability **drops**. Where hiding the pixels hurts the most is where the evidence lived.

MONAI ships `OcclusionSensitivity`, but we build it in a few lines so the mechanism is visible — and so it agrees with Grad-CAM for the *right* reason. On the confounded model, the biggest drop should be at the **ink mark**. Rather than eyeball the two maps, we then *measure* how much of each one's evidence falls on the mark — and only claim agreement if the numbers agree.

In [ ]:
def occlusion_map(net, x01, target, patch=20, stride=16, baseline=0.0):
    net.eval()
    xin = normalize(x01).unsqueeze(0).to(device)
    with torch.no_grad():
        base = Fnn.softmax(net(xin), 1)[0, target].item()
    H, W = IMG_SIZE, IMG_SIZE
    heat = np.zeros((H, W)); cnt = np.zeros((H, W))
    for i in range(0, H - patch + 1, stride):
        for j in range(0, W - patch + 1, stride):
            x2 = xin.clone(); x2[..., i:i+patch, j:j+patch] = baseline
            with torch.no_grad():
                p = Fnn.softmax(net(x2), 1)[0, target].item()
            heat[i:i+patch, j:j+patch] += (base - p); cnt[i:i+patch, j:j+patch] += 1
    return heat / np.maximum(cnt, 1)

x01 = add_ink_mark(to01(xte_s[jt]))
omap = occlusion_map(model_cf, x01, target=1)
fig, ax = plt.subplots(1, 3, figsize=(8, 2.8))
ax[0].imshow(x01.permute(1,2,0).numpy()); ax[0].set_title("tumour + ink", fontsize=9)
gmap = cam_cf(x=normalize(x01).unsqueeze(0).to(device), class_idx=1).squeeze().detach().cpu().numpy()
ax[1].imshow(gmap, cmap="jet", vmin=0, vmax=1)
ax[1].set_title("Grad-CAM", fontsize=9)
ax[2].imshow(omap, cmap="jet"); ax[2].set_title("occlusion (prob drop)", fontsize=9)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
# Don't eyeball it — measure it. Where is each map's evidence: on the ink or on the tissue?
ink = INK_MASK.numpy()
def ink_share(m):
    m = np.clip(m, 0, None)                        # only evidence FOR the class
    return m[ink].sum() / (m.sum() + 1e-8)
print(f"the ink mark covers {ink.mean():.0%} of the tile")
g_share, o_share = ink_share(gmap), ink_share(omap)
print(f"share of Grad-CAM evidence on the ink  : {g_share:.0%}")
print(f"share of occlusion evidence on the ink : {o_share:.0%}")
# "points at the ink" = the map puts clearly more than its fair (area) share of evidence there
g_ink, o_ink = g_share > 2 * ink.mean(), o_share > 2 * ink.mean()
if g_ink and o_ink:
    print("-> both methods point at the ink: two different methods, same verdict = real corroboration.")
elif g_ink != o_ink:
    print("-> the methods DISAGREE. That is a finding too: don't trust either one alone; dig further.")
else:
    print("-> neither method singles out the ink on this tile.")

## 11 · The skeptic's checklist

Apply this to any medical-imaging model before you believe it — it's what the graded assignment asks of yours:

1. Is the heat on **plausible morphology**, or on background, a border, ink, or another artifact?
2. Does it **move sensibly** when you change the target class?
3. Does **occlusion agree** with Grad-CAM, or do they disagree?
4. Would the highlighted region **survive a different stain, scanner, or site**?

5. Would a model that **never saw** the suspected artifact react the same way? (the control)

The confounded model here fails #1 and #4 outright — and because it fails them, its honest-looking accuracy is worthless.

## What this showed, and the hand-forward

- **Transfer learning** beat scratch on scarce real tiles — the lecture's claim, demonstrated.
- A pretrained **DenseNet121**, fine-tuned and ImageNet-normalised, classified real tumour vs. normal tissue.
- A confounded model scored near-perfect **validation** accuracy — and still had learned nothing about tumour biology.
- **Grad-CAM** (with the colour scale fixed so red really means important) showed where the model looked; a **reliance test**, a **flip test with a control**, and a **hand-built occlusion map** confirmed when it was looking in the wrong place.

**Graded notebook next.** You'll run this pipeline on your own slice of pathology and go looking for a confounder *we did not plant* — a real one, which is subtler and not guaranteed to be where you expect. That's the escalation: **toy = engineered and obvious → real = injected and realistic (ink) → graded = find it yourself**, then write the deployment-and-trust discussion.

`Kernel → Restart & Run All` to confirm a clean top-to-bottom run.

In [ ]:
%pip install torch torchvision "monai[pillow]" "medmnist>=3.0" scikit-learn matplotlib